# 🚒 [Mission 2] 지능형 화자 분류 다중 모델 벤치마크 (SOTA 오디오 비교)

본 노트북은 **신고자 vs 119대원 화자 분류(Mission 2)** 문제를 해결하기 위해, 음향 분석 보고서(`speaker_diarization_analysis.md`) 및 글로벌 SOTA 화자 인식 기법을 바탕으로 **6개 모델의 정확도와 효율성(지연시간, 파라미터수, VRAM)**을 동일 데이터셋 위에서 체계적으로 비교하는 통합 벤치마크 시스템입니다.

### 🛡️ 장시간 Colab Pro 무중단 안전장치
1. **Google Drive 실시간 동기화**: 에포크마다 최고 모델 가중치 및 비교표가 드라이브로 즉시 저장됩니다.
2. **Auto-Resume (자동 이어하기)**: 세션이 끊겨 재실행되더라도 이미 학습 완료된 모델은 자동으로 Skip하고 다음 모델부터 진행합니다.
3. **분할 실행 지원**: 가벼운 모델 4종(ReDimNet, ECAPA, CAM++, ResNet)과 무거운 트랜스포머 2종(HuBERT, SSAST)을 분리하여 안전하게 실행할 수 있습니다.

---

### 📊 비교 대상 6개 모델 아키텍처
| 모델 | 분류 | 입력 형태 | 파라미터 | 선정 근거 |
| :--- | :--- | :--- | :---: | :--- |
| **ReDimNet2-B2** | Hybrid (2D+1D Conv + MHA) | Mel-Filterbank | **3.6M** | 🥇 분석 문서 1순위 추천, 초경량 고성능 화자 임베딩 |
| **ECAPA-TDNN** | 1D CNN / Res2Net + SE | Mel-Filterbank | **6.1M** | 🥈 화자 검증 글로벌 표준 모델, 통계적 어텐션 풀링 |
| **CAM++** | Dense TDNN + Masking | Mel-Filterbank | **7.2M** | 🥉 3D-Speaker 국소 특징 보존 및 문맥 마스킹 |
| **AudioResNet-50**| 2D CNN (ImageNet 평균) | Mel-Spectrogram | **23.5M** | 기존 베이스라인 모델 (~90%) |
| **HuBERT-Base** | Self-Supervised Transformer | Raw Waveform (1D) | **95.0M** | 음향 토큰 마스킹 사전학습 SOTA 모델 |
| **SSAST-Tiny** | Audio Vision Transformer | Mel-Spectrogram (ViT) | **6.0M** | ViT 가설 검증용 대조군 |


### [Step 1] GPU 가속기 확인 및 라이브러리 설정


In [ ]:
# GPU 정보 및 VRAM 확인
import torch
import sys, os

print(f"PyTorch 버전: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU 활성화: {gpu_name} (총 VRAM: {vram_gb:.1f} GB)")
else:
    print("⚠️ GPU 가속기가 활성화되지 않았습니다! [런타임] -> [런타임 유형 변경]에서 GPU(A100/V100/T4)를 선택하세요.")

# 의존성 패키지 확인 및 설치
!pip install -q torchaudio transformers scikit-learn tabulate pandas


### [Step 2] 구글 드라이브 마운트 및 실시간 백업 경로 설정


In [ ]:
import os, shutil

# 구글 드라이브 마운트 확인
if not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print("⚠️ 드라이브 자동 마운트 안내: 좌측 📁 파일 아이콘 -> [드라이브 마운트] 버튼을 클릭해 주세요.")

# 실시간 백업 디렉토리 생성
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/DCC/benchmark_results"
os.makedirs(os.path.join(DRIVE_BACKUP_DIR, "checkpoints"), exist_ok=True)
print(f"💾 영구 백업 폴더 준비 완료: {DRIVE_BACKUP_DIR}")

# 벤치마크 모듈 경로 등록
BENCHMARK_MODULE_PATH = "/content/DCC/DCC_Problem/mission2_speaker"
if not os.path.exists(BENCHMARK_MODULE_PATH):
    # 로컬 경로 또는 git clone 경로 자동 탐색
    for candidate in ["/content/DCC_Problem/mission2_speaker", "./mission2_speaker", "."]:
        if os.path.exists(os.path.join(candidate, "benchmark_suite")):
            BENCHMARK_MODULE_PATH = candidate
            break

if BENCHMARK_MODULE_PATH not in sys.path:
    sys.path.insert(0, BENCHMARK_MODULE_PATH)

print(f"📦 벤치마크 모듈 로드 경로: {BENCHMARK_MODULE_PATH}")
from benchmark_suite import MODEL_REGISTRY, UniversalSpeakerDataset, build_model, BenchmarkTrainer, EnsembleEvaluator
print(f"✅ 등록된 모델 목록: {list(MODEL_REGISTRY.keys())}")


### [Step 3] 고속 로컬 SSD 데이터 경로 확인


In [ ]:
import glob

# 데이터 경로 탐색 (/content/data)
DATA_ROOT = "/content/data"
train_wavs = glob.glob(f"{DATA_ROOT}/**/TS_*.wav", recursive=True) + glob.glob(f"{DATA_ROOT}/**/train/**/*.wav", recursive=True)
val_wavs = glob.glob(f"{DATA_ROOT}/**/VS_*.wav", recursive=True) + glob.glob(f"{DATA_ROOT}/**/val/**/*.wav", recursive=True)

print(f"📊 발견된 Train 오디오 파일: {len(train_wavs)}개")
print(f"📊 발견된 Val 오디오 파일   : {len(val_wavs)}개")

# 폴더 경로 자동 설정
if os.path.exists(os.path.join(DATA_ROOT, "train")):
    TRAIN_DIR = os.path.join(DATA_ROOT, "train")
    VAL_DIR = os.path.join(DATA_ROOT, "val")
elif os.path.exists(os.path.join(DATA_ROOT, "대학부 데이터")):
    TRAIN_DIR = os.path.join(DATA_ROOT, "대학부 데이터/Training")
    VAL_DIR = os.path.join(DATA_ROOT, "대학부 데이터/Validation")
else:
    TRAIN_DIR = DATA_ROOT
    VAL_DIR = DATA_ROOT

print(f"📂 Train 디렉토리: {TRAIN_DIR}")
print(f"📂 Val 디렉토리  : {VAL_DIR}")


### [Step 4] 🥇 [Phase 1] 추천 경량 및 베이스라인 모델 4종 실행
* 대상 모델: **`redimnet` (1순위)**, **`ecapa_tdnn` (표준)**, **`campp` (문맥마스킹)**, **`resnet50` (베이스라인)**
* 총 소요 시간: 약 7~8시간 (중간에 끊겨도 드라이브에 안전 백업 & 재실행 시 완료된 모델 자동 Skip)


In [ ]:
from benchmark_suite.benchmark import run_benchmark

# 경량 4대 핵심 모델 벤치마크 실행
PHASE1_MODELS = ["redimnet", "ecapa_tdnn", "campp", "resnet50"]

print("🚀 Phase 1 벤치마크 학습을 시작합니다...")
run_benchmark(
    models_to_run=PHASE1_MODELS,
    epochs=15,               # 수렴에 충분한 15 에폭
    data_root=DATA_ROOT,
    output_dir="/content/results_benchmark",
    skip_completed=True      # 이미 완료된 모델은 자동 스킵!
)


### [Step 5] 🥈 [Phase 2] 트랜스포머 및 ViT 모델 실행 (선택 사항)
* 대상 모델: **`hubert` (대규모 자기지도학습)**, **`ssast` (오디오 ViT)**
* VRAM 절약을 위해 배치가 작게 조정되어 있으며, 상한선 검증용으로 활용됩니다.


In [ ]:
# 트랜스포머 모델 벤치마크 실행 (필요 시 주석 해제 후 실행)
PHASE2_MODELS = ["ssast", "hubert"]

print("🚀 Phase 2 트랜스포머 벤치마크 학습 시작...")
run_benchmark(
    models_to_run=PHASE2_MODELS,
    epochs=10,
    data_root=DATA_ROOT,
    output_dir="/content/results_benchmark",
    skip_completed=True
)


### [Step 6] 🏆 다중 모델 벤치마크 최종 비교 분석 (정확도 vs 지연시간 vs 파라미터)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 결과 CSV 로드
res_path = "/content/results_benchmark/benchmark_results.csv"
if not os.path.exists(res_path):
    res_path = os.path.join(DRIVE_BACKUP_DIR, "benchmark_results.csv")

if os.path.exists(res_path):
    df = pd.read_csv(res_path)
    print("📋 [전체 모델 종합 벤치마크 결과표]")
    display(df)

    # 1. 정확도 vs 파라미터수 (효율성 비교 바 차트)
    fig, ax1 = plt.subplots(figsize=(10, 5))
    models = df["model_name"]
    accs = df["val_accuracy"]
    params = df["params_million"]

    x = np.arange(len(models))
    width = 0.35

    rects1 = ax1.bar(x - width/2, accs, width, label='Val Accuracy (%)', color='#4C72B0')
    ax1.set_ylabel('Accuracy (%)', color='#4C72B0')
    ax1.set_ylim(70, 100)

    ax2 = ax1.twinx()
    rects2 = ax2.bar(x + width/2, params, width, label='Params (M)', color='#55A868', alpha=0.7)
    ax2.set_ylabel('Parameters (Million)', color='#55A868')

    ax1.set_xticks(x)
    ax1.set_xticklabels(models, fontsize=11, fontweight='bold')
    plt.title("Model Performance vs Efficiency Comparison", fontsize=14, fontweight='bold')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.show()

    # 최고 모델 선정 안내
    best_row = df.loc[df['val_accuracy'].idxmax()]
    print(f"🥇 최고 단일 모델: {best_row['model_name']} (정확도: {best_row['val_accuracy']}%, F1: {best_row['macro_f1']})")
else:
    print("아직 완료된 벤치마크 결과 파일이 없습니다.")


### [Step 7] ✨ 이종 아키텍처 결합 Soft Voting 앙상블
학습된 상위 3개 모델(예: `redimnet` + `ecapa_tdnn` + `resnet50`)의 확신도(Soft Probabilities)를 가중 결합하여 최고 성능(93%+)에 도전합니다.


In [ ]:
# 상위 모델 Soft Voting 앙상블 실행
ensemble = EnsembleEvaluator(
    ckpt_dir="/content/results_benchmark/checkpoints",
    data_dir=DATA_ROOT
)

# 앙상블할 모델 선택 (학습 완료된 모델 중)
candidate_models = ["redimnet", "ecapa_tdnn", "campp", "resnet50"]
metrics = ensemble.evaluate_soft_voting(
    model_names=candidate_models,
    val_dir=VAL_DIR
)


### [Step 8] 🎯 최종 제출용 `best_model.pt` 가중치 추출
벤치마크에서 1위를 차지한 모델 가중치를 최종 제출 경로로 복사합니다.


In [ ]:
# 최고 모델을 best_model.pt로 복사
if os.path.exists(res_path):
    df = pd.read_csv(res_path)
    best_name = df.loc[df['val_accuracy'].idxmax()]['model_name']
    src_ckpt = f"/content/results_benchmark/checkpoints/best_{best_name}.pt"
    dst_ckpt = "/content/DCC/best_model.pt"
    
    if os.path.exists(src_ckpt):
        shutil.copy2(src_ckpt, dst_ckpt)
        # 구글 드라이브 최상단에도 복사
        shutil.copy2(src_ckpt, "/content/drive/MyDrive/DCC/best_model.pt")
        print(f"🎉 최종 1위 모델 [{best_name}] 가중치가 {dst_ckpt} 및 구글 드라이브로 복사되었습니다!")
